In [1]:
import matplotlib.pyplot as plt
def lzss_encode(data: bytes, window_size: int = 1024, lookahead_size: int = 15, min_match: int = 3):
    if not data:
        return b''
    
    encoded = bytearray()
    n = len(data)
    pos = 0
    
    while pos < n:
        search_start = max(0, pos - window_size)
        search_buffer = data[search_start:pos]
        max_lookahead = min(lookahead_size, n - pos)
        
        best_offset = 0
        best_length = 0
        
        for length in range(min_match, max_lookahead + 1):
            pattern = data[pos:pos + length]
            offset = search_buffer.rfind(pattern)
            
            if offset != -1:
                actual_offset = len(search_buffer) - offset
                if actual_offset <= window_size:
                    best_offset = actual_offset
                    best_length = length
            else:
                break
        
        if best_length >= min_match:
            # Флаг 0x80 + смещение (2 байта) + длина (1 байт)
            encoded.append(0x80 | ((best_offset >> 8) & 0x7F))
            encoded.append(best_offset & 0xFF)
            encoded.append(best_length)
            pos += best_length
        else:
            # Флаг 0x00 + литерал
            encoded.append(0x00)
            encoded.append(data[pos])
            pos += 1
    
    return bytes(encoded)


def lzss_decode(encoded: bytes):
    decoded = bytearray()
    i = 0
    n = len(encoded)
    
    while i < n:
        flag = encoded[i]
        i += 1
        
        if flag == 0:
            # Литерал
            if i >= n:
                break
            decoded.append(encoded[i])
            i += 1
        else:
            # Ссылка
            if i + 2 > n:
                break
            offset = ((flag & 0x7F) << 8) | encoded[i]
            length = encoded[i + 1]
            i += 2
            
            start_pos = len(decoded) - offset
            for j in range(length):
                decoded.append(decoded[start_pos + j])
    
    return bytes(decoded)

def compress(input_file: str, output_file: str, window_size: int = 1024, lookahead_size: int = 15, min_match: int = 3):
    # Чтение исходного файла
    with open(input_file, 'rb') as f:
        data = f.read()
    
    # Сжатие данных
    compressed_data = lzss_encode(data, window_size, lookahead_size, min_match)
    
    # Запись сжатых данных
    with open(output_file, 'wb') as f:
        f.write(compressed_data)


def decompress(input_file: str, output_file: str):
    # Чтение сжатого файла
    with open(input_file, 'rb') as f:
        compressed_data = f.read()
    
    # Распаковка данных
    decompressed_data = lzss_decode(compressed_data)
    
    # Запись распакованных данных
    with open(output_file, 'wb') as f:
        f.write(decompressed_data)

TEST_FILES = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/enwik7",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test2.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test3.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test4.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test5.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test6.raw",
]

TEST_FILES_COMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/enwik7_comp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test2_comp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test3_comp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test4_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test5_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test6_comp.raw",
]

TEST_FILES_DECOMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/enwik7_decomp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test2_decomp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test3_decomp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test4_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test5_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/lzss/test6_decomp.raw",
]

from pathlib import Path

for file_path in range(len(TEST_FILES)):
    
    original_size = Path(TEST_FILES[file_path]).stat().st_size
    
    print(f"\nФайл: {file_path+1} ({original_size:,} байт)")

    # Сжатие
    compress(TEST_FILES[file_path], TEST_FILES_COMPRESS[file_path])
    compressed_size = Path(TEST_FILES_COMPRESS[file_path]).stat().st_size
    
    # Распаковка
    decompress(TEST_FILES_COMPRESS[file_path], TEST_FILES_DECOMPRESS[file_path])
    decompressed_size = Path(TEST_FILES_DECOMPRESS[file_path]).stat().st_size
    
    # Проверка целостности
    if decompressed_size == original_size:
        print("correct")
    else:
        print("fail")
    
    # Статистика
    ratio = (original_size / compressed_size) if compressed_size > 0 else 0
    print(f"Исходный размер: {original_size}\n")
    print(f"После сжатия: {compressed_size}\n")
    print(f"После распаковки: {decompressed_size}\n")
    print(f"Коэффициент сжатия: {ratio}\n")


Файл: 1 (10,000,000 байт)
correct
Исходный размер: 10000000

После сжатия: 9864125

После распаковки: 10000000

Коэффициент сжатия: 1.013774663236729


Файл: 2 (371,816 байт)
correct
Исходный размер: 371816

После сжатия: 252799

После распаковки: 371816

Коэффициент сжатия: 1.470796957266445


Файл: 3 (1,104,440 байт)
correct
Исходный размер: 1104440

После сжатия: 1209277

После распаковки: 1104440

Коэффициент сжатия: 0.9133060498132355


Файл: 4 (1,532,562 байт)
correct
Исходный размер: 1532562

После сжатия: 1521103

После распаковки: 1532562

Коэффициент сжатия: 1.0075333491551854


Файл: 5 (2,001,012 байт)
correct
Исходный размер: 2001012

После сжатия: 2537206

После распаковки: 2001012

Коэффициент сжатия: 0.7886675342877165


Файл: 6 (1,806,348 байт)
correct
Исходный размер: 1806348

После сжатия: 3414002

После распаковки: 1806348

Коэффициент сжатия: 0.5290998657880107

